# 2025-26 Target-Prompt Experiments — Name-as-Written, Don't-Guess, Reframe, Format

**Objective:** Freeze the sentiment-target verifier's prompt (#91 PR B) by running
candidate lines through the production harness (`classify_target_cases(prompt_builder=...)`)
against the 111-case target suite. Decision rule, per the ticket amendment
(2026-09-06): every variant is scored on all five category floors through
`verdict_correct` (alias-map resolution, as production consumes it), and
**`true_toward` is the veto** — adopt nothing that moves it down a case.
Deliverable: this notebook + per-variant artifacts under
`data/2025-26/reference/prompt_experiments/target_*.json`, an adopt/reject
verdict per line, and the frozen prompt applied to `pipeline/targets.py`
(same PR, separate commit). Floors and `known_miss` flags in
`tests/eval/target_cases.yaml` are re-pinned only after the freeze.

**This notebook answers:**

- §1 — Control: does a fresh run of `v0-draft` reproduce the pinned flags
  (post config v4.5, which resolved the three alias-gap cases)?
- §2 — Variant A, *name as written* + *don't guess*: two independent lines
  with no interaction risk. Four misses are the model formalizing a name the
  alias map already resolves as written; two are guesses at unnamed players.
- §3 — Variant B, A + a *reframed question* that makes the non-player answer
  first-class. The mention-read-as-target bucket (~15 misses) — the v0 prompt
  already says "not merely mentioned" in words and the model ignores it, so
  this changes the form of the question, not the volume of instruction.
- §4 — Variant C, best-so-far + *no explanation*: format hygiene. Zero scoring
  effect since the first-object parser fix, so the win condition is output
  tokens and the end of the "let me reconsider" pattern, at every floor held.
- §5 — Reserve: one run of `v0-draft` on Sonnet, only if §3 leaves the mention
  bucket where it was — prompt ceiling or model ceiling?
- §6 — Freeze candidate, 3-run confirmed; the stable-flip table drives the re-pin.
- §7 — Verdict: per-line adopt/reject with suite evidence; the frozen prompt.

## Load Data

Guardrails:

- **Every variant run costs 111 sync Messages API calls** (Haiku @ temp 0,
  ~$0.02/run). Results persist to `EXP_DIR` keyed by a sha of the built prompt
  (`target_` prefix — the directory is shared with 06): re-executing the
  notebook re-spends nothing until a prompt actually changes.
- **Variants live here, not in `pipeline/targets.py`.** Production
  `build_target_prompt` is imported untouched as the §1 control; only the
  §7-frozen winner lands in the module. The pytest suite keeps measuring
  production throughout.
- **Scoring is the harness's own:** verifier model params, parser, and
  `verdict_correct` (alias-map resolution). The flips table scores on
  resolved correctness — a name-form change from `Luguentz Dort` to `Dort`
  shows as a flip; a change between two wrong names does not. Lesson from
  06's addendum, where `flips()` only watched one axis.
- Single-run deltas are readable because the 09-06 baseline had one flaky
  case across 3 runs (`receipt-m23`); the freeze candidate still gets 3 runs.

In [1]:
import hashlib
import json
import os
import re
import sys
from collections.abc import Callable
from pathlib import Path

# Bootstrap: run from anywhere (see 01–07 — walk up to the repo root, chdir).
_root = Path.cwd()
while not (_root / "pyproject.toml").exists() and _root != _root.parent:
    _root = _root.parent
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import polars as pl

from pipeline.targets import (
    TARGET_MAX_TOKENS,
    TARGET_MODEL,
    TARGET_PROMPT_VERSION,
    TARGET_TEMPERATURE,
    build_target_prompt,
    classify_target_cases,
    load_target_cases,
    load_target_floors,
    target_accuracy_by_category,
    verdict_correct,
)
from utils.paths import get_data_dir
from utils.player_config import (
    build_alias_to_player_map,
    load_player_config_version,
    resolve_sentiment_player,
)
from utils.season_config import get_active_season

SEASON = "2025-26"  # pinned: this notebook is season-scoped (notebooks/2025-26/)
assert get_active_season() == SEASON, "active season flipped — review before re-running"
assert load_player_config_version() == "4.5", "alias map moved — re-derive the pinned picture"

EXP_DIR = get_data_dir(season=SEASON) / "reference" / "prompt_experiments"
EXP_DIR.mkdir(parents=True, exist_ok=True)

CASES = load_target_cases()
FLOORS = load_target_floors()
ALIAS_MAP = build_alias_to_player_map()
CASE_BY_ID = {c.id: c for c in CASES}

# The pinned picture, derived from the known_miss flags (a case is
# expected-correct iff it isn't flagged). Fresh runs compare against this.
PINNED = {
    cat: (
        sum(1 for c in CASES if c.category == cat and not c.known_miss),
        sum(1 for c in CASES if c.category == cat),
    )
    for cat in FLOORS
}

assert os.environ.get("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY not set"
print(
    f"{len(CASES)} cases, verifier {TARGET_PROMPT_VERSION} on {TARGET_MODEL} "
    f"@ temp {TARGET_TEMPERATURE}, max_tokens {TARGET_MAX_TOKENS}"
)
print(
    pl.DataFrame(
        [
            {"category": cat, "pinned": f"{c}/{t}", "floor": FLOORS[cat]}
            for cat, (c, t) in sorted(PINNED.items())
        ]
    )
)

111 cases, verifier v0-draft on claude-haiku-4-5-20251001 @ temp 0.0, max_tokens 75
shape: (5, 3)
┌────────────────────┬────────┬───────┐
│ category           ┆ pinned ┆ floor │
│ ---                ┆ ---    ┆ ---   │
│ str                ┆ str    ┆ f64   │
╞════════════════════╪════════╪═══════╡
│ non_player         ┆ 8/11   ┆ 0.63  │
│ readmit_affirm     ┆ 7/9    ┆ 0.66  │
│ subject_not_target ┆ 24/38  ┆ 0.63  │
│ true_toward        ┆ 49/52  ┆ 0.86  │
│ wrong_player       ┆ 1/1    ┆ 1.0   │
└────────────────────┴────────┴───────┘


In [2]:
# Variant runner + scoring helpers used by every section below.
pl.Config.set_tbl_rows(60)
pl.Config.set_fmt_str_lengths(64)
pl.Config.set_tbl_width_chars(140)

Builder = Callable[[str, str], str]


def prompt_sha(prompt_builder: Builder) -> str:
    """A variant's identity: sha of the prompt built for a sentinel body."""
    canary = prompt_builder("__CANARY__", "neg") + prompt_builder("__CANARY__", "pos")
    return hashlib.sha256(canary.encode()).hexdigest()[:16]


def run_variant(
    name: str, prompt_builder: Builder, run: int = 1, model: str = TARGET_MODEL
) -> dict[str, dict]:
    """Verify all cases under a variant, cached on (name, run, prompt sha, model).

    Bump `run` for an independent repeat of the same prompt (stability
    checks); a changed prompt under the same name re-runs automatically.
    """
    sha = prompt_sha(prompt_builder)
    artifact = EXP_DIR / f"target_{name}_run{run}.json"
    if artifact.exists():
        payload = json.loads(artifact.read_text())
        if payload["prompt_sha"] == sha and payload["model"] == model:
            return payload["results"]
        print(f"{artifact.name}: prompt/model changed — re-running")
    if model == TARGET_MODEL:
        results = classify_target_cases(CASES, prompt_builder=prompt_builder)
    else:
        results = classify_on_model(prompt_builder, model)
    artifact.write_text(
        json.dumps({"prompt_sha": sha, "model": model, "results": results}, indent=1)
    )
    invalid = sum(1 for r in results.values() if not r["valid"])
    print(f"{artifact.name}: written ({invalid} invalid parses)")
    return results


def classify_on_model(prompt_builder: Builder, model: str) -> dict[str, dict]:
    """§5 reserve only: the harness runner with the model swapped.

    Sonnet 5 rejects sampling params (temperature) with a 400 and runs
    adaptive thinking unless disabled; thinking off + default sampling is
    the closest analog to the verifier's temp-0 Haiku call.
    """
    import anthropic

    from pipeline.targets import parse_target_response

    client = anthropic.Anthropic()
    results = {}
    for case in CASES:
        response = client.messages.create(
            model=model,
            max_tokens=TARGET_MAX_TOKENS,
            thinking={"type": "disabled"},
            messages=[{"role": "user", "content": prompt_builder(case.text, case.sentiment)}],
        )
        text = next(b.text for b in response.content if b.type == "text")
        results[case.id] = {
            **parse_target_response(text), "raw": text, "stop_reason": response.stop_reason,
        }
    return results


def correct(results: dict[str, dict], case_id: str) -> bool:
    """Resolved correctness, exactly as the suite scores it."""
    return verdict_correct(results[case_id], CASE_BY_ID[case_id].expected_target, ALIAS_MAP)


def resolved(result: dict) -> str | None:
    """What production would attribute from this verdict (None if unresolvable)."""
    return resolve_sentiment_player(result["t"], ALIAS_MAP) if result["valid"] else None


def category_table(results: dict[str, dict]) -> pl.DataFrame:
    """Per-category verdict accuracy vs the pinned picture and the floor."""
    tallies = target_accuracy_by_category(CASES, results, ALIAS_MAP)
    return pl.DataFrame(
        [
            {
                "category": cat,
                "acc": f"{c}/{t}",
                "pinned": f"{pc}/{pt}",
                "delta": c - pc,
                "floor_ok": c / t >= FLOORS[cat],
            }
            for cat, (c, t) in sorted(tallies.items())
            for pc, pt in [PINNED[cat]]
        ]
    ).sort("delta")


FLIP_SCHEMA = {
    "id": str, "category": str, "expected": str,
    "ref": str, "got": str, "flip": str, "text": str,
}


def flips(results: dict[str, dict], against: dict[str, dict]) -> pl.DataFrame:
    """Cases whose resolved correctness changed vs a reference run.

    `ref`/`got` show the raw verdict so a name-form change is visible;
    `fixed` rows are the variant's wins, `BROKEN` rows are its cost.
    """
    rows = []
    for case in CASES:
        if correct(results, case.id) == correct(against, case.id):
            continue
        rows.append(
            {
                "id": case.id, "category": case.category,
                "expected": str(case.expected_target),
                "ref": str(against[case.id]["t"]), "got": str(results[case.id]["t"]),
                "flip": "fixed" if correct(results, case.id) else "BROKEN",
                "text": case.text[:60].replace("\n", " "),
            }
        )
    return pl.DataFrame(rows, schema=FLIP_SCHEMA).sort(["flip", "category", "id"])


def unresolved(results: dict[str, dict]) -> pl.DataFrame:
    """Valid, non-null verdicts that resolve to no tracked player — the alias-gap class."""
    rows = [
        {"id": c.id, "got": r["t"], "expected": str(c.expected_target)}
        for c in CASES
        for r in [results[c.id]]
        if r["valid"] and r["t"] is not None and resolved(r) is None
    ]
    return pl.DataFrame(rows, schema={"id": str, "got": str, "expected": str})


_JSON_ONLY_RE = re.compile(r"^\s*\{.*\}\s*$", re.DOTALL)


def format_table(runs: dict[str, dict[str, dict]]) -> pl.DataFrame:
    """Output-format hygiene per run: parse validity, truncation, prose, length."""
    return pl.DataFrame(
        [
            {
                "run": name,
                "invalid": sum(1 for r in res.values() if not r["valid"]),
                "truncated": sum(1 for r in res.values() if r["stop_reason"] == "max_tokens"),
                "not_bare_json": sum(1 for r in res.values() if not _JSON_ONLY_RE.match(r["raw"])),
                "mean_chars": round(sum(len(r["raw"]) for r in res.values()) / len(res), 1),
            }
            for name, res in runs.items()
        ]
    )

## 1. Control — fresh run of `v0-draft`

`PINNED` above is the 09-06 3-run picture reconstructed from the `known_miss`
flags, with the three alias-gap cases (`receipt-m29`, `random-m02`,
`random-m03`) already un-flagged by the config v4.5 commit — their recorded
verdicts resolve now. A fresh run that matches it makes every later
single-run delta readable as prompt effect. Per-case drift prints below; the
only expected entry is the flaky `receipt-m23`. This run becomes the flip-table
reference for every variant.

In [3]:
# §1 — control run: production build_target_prompt, untouched.
baseline = run_variant("baseline", build_target_prompt)
print(category_table(baseline))

drift = pl.DataFrame(
    [
        {
            "id": c.id, "category": c.category, "expected": str(c.expected_target),
            "got": str(baseline[c.id]["t"]),
            "drift": "surprise pass" if c.known_miss else "fresh miss",
            "text": c.text[:60].replace("\n", " "),
        }
        for c in CASES
        if correct(baseline, c.id) == c.known_miss
    ],
    schema={"id": str, "category": str, "expected": str, "got": str, "drift": str, "text": str},
)
print(f"\nper-case drift vs pinned flags: {drift.height} case(s)")
print(drift)
print("\nunresolved verdicts (alias-gap class):")
print(unresolved(baseline))

shape: (5, 5)
┌────────────────────┬───────┬────────┬───────┬──────────┐
│ category           ┆ acc   ┆ pinned ┆ delta ┆ floor_ok │
│ ---                ┆ ---   ┆ ---    ┆ ---   ┆ ---      │
│ str                ┆ str   ┆ str    ┆ i64   ┆ bool     │
╞════════════════════╪═══════╪════════╪═══════╪══════════╡
│ non_player         ┆ 8/11  ┆ 8/11   ┆ 0     ┆ true     │
│ readmit_affirm     ┆ 7/9   ┆ 7/9    ┆ 0     ┆ true     │
│ subject_not_target ┆ 24/38 ┆ 24/38  ┆ 0     ┆ true     │
│ true_toward        ┆ 49/52 ┆ 49/52  ┆ 0     ┆ true     │
│ wrong_player       ┆ 1/1   ┆ 1/1    ┆ 0     ┆ true     │
└────────────────────┴───────┴────────┴───────┴──────────┘

per-case drift vs pinned flags: 0 case(s)
shape: (0, 6)
┌─────┬──────────┬──────────┬─────┬───────┬──────┐
│ id  ┆ category ┆ expected ┆ got ┆ drift ┆ text │
│ --- ┆ ---      ┆ ---      ┆ --- ┆ ---   ┆ ---  │
│ str ┆ str      ┆ str      ┆ str ┆ str   ┆ str  │
╞═════╪══════════╪══════════╪═════╪═══════╪══════╡
└─────┴──────────┴───────

## 2. Variant A — name as written + don't guess

Two independent lines, one variant; regressions are attributed by case shape
and the lines split into separate runs only if that isn't obvious.

- **Name as written** is the biggest lever and isn't about judgment: the alias
  map is built from r/NBA spellings and `brooks`, `dort`, `cmb` all resolve
  today, but the model formalizes (`Scottie Brooks`, `Luguentz Dort`,
  `Donovan Clingan` for CMB). Expected flips: `receipt-m33`, `nullp-m17`,
  `random-m11`, plus `random-m02` (already passing via v4.5 — should stay).
  Watch: possessives (`luka's` does not resolve) — the `unresolved` table
  below is the tell.
- **Don't guess** targets `nullp-m32` (Jokic for "the ultimate basketball
  genius") and `random-m28` (Shai for "wemby killer"). Production value is
  larger than two cases: a guessed target re-files a receipt under an innocent
  player. Watch: `readmit_affirm` (one miss fails the floor) and
  `receipt-m17` (Wemby via "8 extra inches"), the only described-but-unnamed
  guard in the suite.

The factory below composes variant prompts in the production shape and is
asserted byte-identical to `build_target_prompt` at defaults.

In [4]:
# §2 — prompt factory (asserted faithful to production) + Variant A.
from pipeline.targets import TARGET_PROMPT_TEMPLATE

SENTIMENT_WORDS = {"pos": "positive", "neg": "negative"}

PROD_ASK = (
    "This r/NBA comment was labeled {sentiment_word}. Name the NBA player that "
    "{sentiment_word} sentiment is directed at."
)
PROD_TARGET_LINE = (
    "The target is the player being praised or criticized - not a player who is "
    "merely mentioned, sympathized with, or the subject of someone else's decision."
)
PROD_NULL_LINE = (
    "If the sentiment is directed at a non-player (front office, coach, referees, "
    "fans, media) or at no one in particular, answer null."
)
PROD_TAIL = 'Respond ONLY with JSON: {{"t":"Player Name"|null,"c":0.0-1.0}}'


def make_builder(
    ask: str = PROD_ASK,
    lines: tuple[str, ...] = (PROD_TARGET_LINE, PROD_NULL_LINE),
    tail: str = PROD_TAIL,
) -> Builder:
    """Variant prompts in the production shape; defaults reproduce it exactly."""
    template = "\n".join([ask, *lines]) + "\n\nComment: {comment_body}\n\n" + tail

    def builder(comment_body: str, sentiment: str) -> str:
        return template.format(
            sentiment_word=SENTIMENT_WORDS[sentiment], comment_body=comment_body
        )

    return builder


assert make_builder()("__x__", "neg") == build_target_prompt("__x__", "neg"), "factory drifted"
assert make_builder()("__x__", "pos") == TARGET_PROMPT_TEMPLATE.format(
    sentiment_word="positive", comment_body="__x__"
)

NAME_LINE = (
    "Write the player's name as it appears in the comment (nickname or surname is "
    "fine; no possessives), or their full name if they are described but not named."
)
GUESS_LINE = (
    "If no player is named or unambiguously described in the comment, answer null "
    "- do not guess."
)

variant_a = run_variant(
    "a_name_noguess", make_builder(lines=(PROD_TARGET_LINE, PROD_NULL_LINE, NAME_LINE, GUESS_LINE))
)
print(category_table(variant_a))
print(flips(variant_a, baseline))
print("\nunresolved verdicts:")
print(unresolved(variant_a))
_m17 = variant_a["receipt-m17"]
print(f"\nreceipt-m17 (described, not named): expected Wembanyama, got {_m17['t']!r}")

shape: (5, 5)
┌────────────────────┬───────┬────────┬───────┬──────────┐
│ category           ┆ acc   ┆ pinned ┆ delta ┆ floor_ok │
│ ---                ┆ ---   ┆ ---    ┆ ---   ┆ ---      │
│ str                ┆ str   ┆ str    ┆ i64   ┆ bool     │
╞════════════════════╪═══════╪════════╪═══════╪══════════╡
│ readmit_affirm     ┆ 5/9   ┆ 7/9    ┆ -2    ┆ false    │
│ non_player         ┆ 8/11  ┆ 8/11   ┆ 0     ┆ true     │
│ wrong_player       ┆ 1/1   ┆ 1/1    ┆ 0     ┆ true     │
│ true_toward        ┆ 51/52 ┆ 49/52  ┆ 2     ┆ true     │
│ subject_not_target ┆ 29/38 ┆ 24/38  ┆ 5     ┆ true     │
└────────────────────┴───────┴────────┴───────┴──────────┘
shape: (13, 7)
┌─────────────┬────────────────────┬───────────────────┬─────────────────────────┬─────────────────┬────────┬──────────────────────────────┐
│ id          ┆ category           ┆ expected          ┆ ref                     ┆ got             ┆ flip   ┆ text                         │
│ ---         ┆ ---                ┆ ---

### 2b. Splitting the lines

Variant A's flip table is net +9 / −4, but the four breaks land on
`readmit_affirm` (7→5, floor fail) and aren't attributable by shape alone:
`random-m22` (`Luka Flopbitch` — the comment's "Fluka Flopbitch" copied as
written) is the name line; the two new nulls (`nullp-m33` Bronny in a Kings
fan post, `random-m21` a bare list of three Spurs) read like the don't-guess
line raising the null bar on borderline readmit rows; `receipt-m13`
(`AD` for an unnamed "him") could be either. So each line runs alone.

In [5]:
# §2b — each Variant A line alone, against the same control.
a_name = run_variant("a1_name", make_builder(lines=(PROD_TARGET_LINE, PROD_NULL_LINE, NAME_LINE)))
a_guess = run_variant("a2_noguess", make_builder(lines=(PROD_TARGET_LINE, PROD_NULL_LINE, GUESS_LINE)))

for label, res in [("name as written alone", a_name), ("don't guess alone", a_guess)]:
    print(f"===== {label} =====")
    print(category_table(res))
    print(flips(res, baseline).select("id", "category", "flip", "ref", "got"))
    print("unresolved:", unresolved(res).select("id", "got").rows())
    print()

===== name as written alone =====
shape: (5, 5)
┌────────────────────┬───────┬────────┬───────┬──────────┐
│ category           ┆ acc   ┆ pinned ┆ delta ┆ floor_ok │
│ ---                ┆ ---   ┆ ---    ┆ ---   ┆ ---      │
│ str                ┆ str   ┆ str    ┆ i64   ┆ bool     │
╞════════════════════╪═══════╪════════╪═══════╪══════════╡
│ readmit_affirm     ┆ 4/9   ┆ 7/9    ┆ -3    ┆ false    │
│ wrong_player       ┆ 1/1   ┆ 1/1    ┆ 0     ┆ true     │
│ non_player         ┆ 9/11  ┆ 8/11   ┆ 1     ┆ true     │
│ true_toward        ┆ 50/52 ┆ 49/52  ┆ 1     ┆ true     │
│ subject_not_target ┆ 29/38 ┆ 24/38  ┆ 5     ┆ true     │
└────────────────────┴───────┴────────┴───────┴──────────┘
shape: (16, 5)
┌─────────────┬────────────────────┬────────┬───────────────────┬────────────────┐
│ id          ┆ category           ┆ flip   ┆ ref               ┆ got            │
│ ---         ┆ ---                ┆ ---    ┆ ---               ┆ ---            │
│ str         ┆ str                ┆ st

### 2c. Terser rewrites

Two things the split shows. The don't-guess line as worded is vetoed outright:
it nulls plainly named players (`random-m04` "I LOVE AJAY MITCHELL",
`random-m15` Giannis) — `true_toward` −3 — because "named or unambiguously
described … do not guess" reads as *when in doubt, null*. And the three
readmit nulls (`nullp-m04`, `nullp-m33`, `random-m21`) appear under **both**
lines alone — a length effect, or the shared "named" wording? Both lines
say it, and the readmit rows are exactly the ones the pass-1 classifier
itself declined, so a null nudge lands there first. The name line's own breaks are copy
artifacts — `Luka Flopbitch`, `Luka and Jokic`, `He`.

So: terser lines that never say "named". The name line asks for one player
and no possessives; the guess line states the prohibition positively — a
player who isn't in the comment can't be the answer — with no null push.

In [6]:
# §2c — terser rewrites, each alone and bundled.
NAME_LINE_V2 = (
    "Spell the name the way the comment does (surname or nickname is fine; "
    "one player, no possessives)."
)
GUESS_LINE_V2 = "Never name a player who does not appear in the comment."

name_v2 = run_variant("a1_name_v2", make_builder(lines=(PROD_TARGET_LINE, PROD_NULL_LINE, NAME_LINE_V2)))
guess_v2 = run_variant("a2_noguess_v2", make_builder(lines=(PROD_TARGET_LINE, PROD_NULL_LINE, GUESS_LINE_V2)))
variant_a2 = run_variant(
    "a_v2", make_builder(lines=(PROD_TARGET_LINE, PROD_NULL_LINE, NAME_LINE_V2, GUESS_LINE_V2))
)

for label, res in [("name v2 alone", name_v2), ("guess v2 alone", guess_v2), ("A v2 bundled", variant_a2)]:
    print(f"===== {label} =====")
    print(category_table(res))
    print(flips(res, baseline).select("id", "category", "flip", "ref", "got"))
    print("unresolved:", unresolved(res).select("id", "got").rows())
    print()

===== name v2 alone =====
shape: (5, 5)
┌────────────────────┬───────┬────────┬───────┬──────────┐
│ category           ┆ acc   ┆ pinned ┆ delta ┆ floor_ok │
│ ---                ┆ ---   ┆ ---    ┆ ---   ┆ ---      │
│ str                ┆ str   ┆ str    ┆ i64   ┆ bool     │
╞════════════════════╪═══════╪════════╪═══════╪══════════╡
│ readmit_affirm     ┆ 5/9   ┆ 7/9    ┆ -2    ┆ false    │
│ subject_not_target ┆ 23/38 ┆ 24/38  ┆ -1    ┆ false    │
│ wrong_player       ┆ 1/1   ┆ 1/1    ┆ 0     ┆ true     │
│ non_player         ┆ 9/11  ┆ 8/11   ┆ 1     ┆ true     │
│ true_toward        ┆ 52/52 ┆ 49/52  ┆ 3     ┆ true     │
└────────────────────┴───────┴────────┴───────┴──────────┘
shape: (13, 5)
┌─────────────┬────────────────────┬────────┬───────────────────┬────────────────┐
│ id          ┆ category           ┆ flip   ┆ ref               ┆ got            │
│ ---         ┆ ---                ┆ ---    ┆ ---               ┆ ---            │
│ str         ┆ str                ┆ str    ┆ s

## 3. Variant B — the reframed question

§2's verdict first. **Don't-guess v2 alone is the §2 winner**: every floor
holds, +9 / −2, `true_toward` flat (the veto), `readmit_affirm` up one, and
it captures most of the name-as-written win by itself — "appear in the
comment" nudges the spelling toward the text (`Scottie Brooks` → `Brooks`,
`random-m05` → `Cade Cunningham`). Its two breaks are outside the prompt's
reach: `random-m06` is a model-invented accent (`Donté` for "DONTEEEEEE")
that the alias map doesn't carry, and `receipt-m35` is a two-target row
("Mobley and Allen") going null under the single-target contract — the same
class as `random-m05`, which flipped the other way. **The name line is
rejected in both wordings**: it tells the model to pick a name from the text,
which is the mention-as-target failure the suite exists to catch
(`nullp-m01`, `receipt-m13`, `random-m23` regress), and `Luka Flopbitch`
survived the rewrite. The readmit nulls under the v1 lines belonged to the
"named or described" wording, not to prompt length — they vanish under v2.

Now the mention bucket. After don't-guess, `subject_not_target` sits at 29/38;
the nine left are mostly the Luka trade cell and dunk-victim rows, where the
v0 prompt already says "not merely mentioned" and the model names Luka on 3
of 8 trade rows while getting the other 5. More words of judgment is the #62
failure mode. The candidate changes the *form*: the question itself offers the
non-player answers as first-class options, and it **replaces** the ask and
the null line rather than adding to them — same token budget. Runs alone
(attribution) and on top of don't-guess v2 (the candidate).

In [7]:
# §3 — reframed ask, replacing PROD_ASK + PROD_NULL_LINE; alone and on top of don't-guess v2.
REFRAME_ASK = (
    "This r/NBA comment was labeled {sentiment_word}. Is that {sentiment_word} "
    "sentiment aimed at an NBA player, or at a team, front office, coach, referees, "
    "a decision, or a play? If at a player, name them; otherwise answer null."
)

reframe_alone = run_variant("b1_reframe", make_builder(ask=REFRAME_ASK, lines=(PROD_TARGET_LINE,)))
variant_b = run_variant(
    "b_reframe_noguess", make_builder(ask=REFRAME_ASK, lines=(PROD_TARGET_LINE, GUESS_LINE_V2))
)

for label, res, ref in [
    ("reframe alone (vs control)", reframe_alone, baseline),
    ("B = reframe + don't-guess v2 (vs don't-guess v2)", variant_b, guess_v2),
]:
    print(f"===== {label} =====")
    print(category_table(res))
    print(flips(res, ref).select("id", "category", "flip", "ref", "got"))
    print("unresolved:", unresolved(res).select("id", "got").rows())
    print()

===== reframe alone (vs control) =====
shape: (5, 5)
┌────────────────────┬───────┬────────┬───────┬──────────┐
│ category           ┆ acc   ┆ pinned ┆ delta ┆ floor_ok │
│ ---                ┆ ---   ┆ ---    ┆ ---   ┆ ---      │
│ str                ┆ str   ┆ str    ┆ i64   ┆ bool     │
╞════════════════════╪═══════╪════════╪═══════╪══════════╡
│ readmit_affirm     ┆ 5/9   ┆ 7/9    ┆ -2    ┆ false    │
│ true_toward        ┆ 48/52 ┆ 49/52  ┆ -1    ┆ true     │
│ wrong_player       ┆ 1/1   ┆ 1/1    ┆ 0     ┆ true     │
│ non_player         ┆ 10/11 ┆ 8/11   ┆ 2     ┆ true     │
│ subject_not_target ┆ 30/38 ┆ 24/38  ┆ 6     ┆ true     │
└────────────────────┴───────┴────────┴───────┴──────────┘
shape: (19, 5)
┌─────────────┬────────────────────┬────────┬─────────────────────────┬──────────────────┐
│ id          ┆ category           ┆ flip   ┆ ref                     ┆ got              │
│ ---         ┆ ---                ┆ ---    ┆ ---                     ┆ ---              │
│ str     

### 3b. Was it the reframing, or the list?

The reframe is the strongest lever yet on the bucket that matters —
`subject_not_target` 29→34/38 on top of don't-guess, and the Luka trade rows
(`receipt-m48/m50/m52`, the ticket's acceptance exemplar) all go null — but
it fails the veto and two floors. "Or at a team" competes with the rubric's
*any sentiment on a named player counts*, and the model takes "team" on the
Kings and Spurs readmit rows (5/9); `true_toward` −1; and in combination
`receipt-m17` (Wemby described, not named) goes null once "appear in the
comment" sits next to "name them" — `wrong_player` 0/1. One format wart too:
"name them" beside the `"Player Name"` placeholder makes the model echo the
placeholder literally on 4–5 rows, which scores as an accidental null.

What the v0 null line lacks is not form but *coverage*: it lists front office,
coach, referees, fans, media — none of the trade-decision or dunk-victim
shapes. So the minimal test: v0 structure kept, the null line's list extended
with team / decision / play, on top of don't-guess v2.

In [8]:
# §3b — v0 form, extended null list, on top of don't-guess v2.
NULL_LINE_V2 = (
    "If the sentiment is directed at a team, front office, coach, referees, fans, "
    "media, a decision, or a play rather than at a player, or at no one in "
    "particular, answer null."
)

variant_b2 = run_variant(
    "b2_nulllist_noguess", make_builder(lines=(PROD_TARGET_LINE, NULL_LINE_V2, GUESS_LINE_V2))
)
print(category_table(variant_b2))
print(flips(variant_b2, guess_v2).select("id", "category", "flip", "ref", "got"))
print("unresolved:", unresolved(variant_b2).select("id", "got").rows())

shape: (5, 5)
┌────────────────────┬───────┬────────┬───────┬──────────┐
│ category           ┆ acc   ┆ pinned ┆ delta ┆ floor_ok │
│ ---                ┆ ---   ┆ ---    ┆ ---   ┆ ---      │
│ str                ┆ str   ┆ str    ┆ i64   ┆ bool     │
╞════════════════════╪═══════╪════════╪═══════╪══════════╡
│ non_player         ┆ 8/11  ┆ 8/11   ┆ 0     ┆ true     │
│ readmit_affirm     ┆ 7/9   ┆ 7/9    ┆ 0     ┆ true     │
│ true_toward        ┆ 49/52 ┆ 49/52  ┆ 0     ┆ true     │
│ wrong_player       ┆ 1/1   ┆ 1/1    ┆ 0     ┆ true     │
│ subject_not_target ┆ 31/38 ┆ 24/38  ┆ 7     ┆ true     │
└────────────────────┴───────┴────────┴───────┴──────────┘
shape: (8, 5)
┌─────────────┬────────────────────┬────────┬───────────────────────┬─────────────────────────┐
│ id          ┆ category           ┆ flip   ┆ ref                   ┆ got                     │
│ ---         ┆ ---                ┆ ---    ┆ ---                   ┆ ---                     │
│ str         ┆ str                

### 3c. Reframe, second cut

The list isn't it: 3b holds every floor and reaches 31/38, but the Luka trade
rows don't move, and against don't-guess alone it's 4 fixed / 4 broken on
borderline rows that wobble between runs. The trade cell responds to the
*question's form*, not to the list — so one more cut of the reframe, since
both of §3's floor failures had identifiable causes. "Team" comes out of the
option list (that's what took the Kings and Spurs readmit rows); "name them"
becomes "answer with the player's name" (the placeholder echo). Alone and
on top of don't-guess v2, because `receipt-m17` only broke in combination.

In [9]:
# §3c — reframe v2: no "team" option, no "name them"; alone and with don't-guess v2.
REFRAME_ASK_V2 = (
    "This r/NBA comment was labeled {sentiment_word}. Is that {sentiment_word} "
    "sentiment aimed at an NBA player, or at a front office, coach, referees, fans, "
    "media, a decision, or a play? If at a player, answer with the player's name; "
    "otherwise answer null."
)

reframe2_alone = run_variant("b3_reframe2", make_builder(ask=REFRAME_ASK_V2, lines=(PROD_TARGET_LINE,)))
variant_b3 = run_variant(
    "b3_reframe2_noguess", make_builder(ask=REFRAME_ASK_V2, lines=(PROD_TARGET_LINE, GUESS_LINE_V2))
)

for label, res, ref in [
    ("reframe v2 alone (vs control)", reframe2_alone, baseline),
    ("B3 = reframe v2 + don't-guess v2 (vs don't-guess v2)", variant_b3, guess_v2),
]:
    print(f"===== {label} =====")
    print(category_table(res))
    print(flips(res, ref).select("id", "category", "flip", "ref", "got"))
    print("unresolved:", unresolved(res).select("id", "got").rows())
    print()

===== reframe v2 alone (vs control) =====
shape: (5, 5)
┌────────────────────┬───────┬────────┬───────┬──────────┐
│ category           ┆ acc   ┆ pinned ┆ delta ┆ floor_ok │
│ ---                ┆ ---   ┆ ---    ┆ ---   ┆ ---      │
│ str                ┆ str   ┆ str    ┆ i64   ┆ bool     │
╞════════════════════╪═══════╪════════╪═══════╪══════════╡
│ readmit_affirm     ┆ 5/9   ┆ 7/9    ┆ -2    ┆ false    │
│ true_toward        ┆ 48/52 ┆ 49/52  ┆ -1    ┆ true     │
│ wrong_player       ┆ 1/1   ┆ 1/1    ┆ 0     ┆ true     │
│ non_player         ┆ 10/11 ┆ 8/11   ┆ 2     ┆ true     │
│ subject_not_target ┆ 31/38 ┆ 24/38  ┆ 7     ┆ true     │
└────────────────────┴───────┴────────┴───────┴──────────┘
shape: (16, 5)
┌─────────────┬────────────────────┬────────┬─────────────────────────┬──────────────────┐
│ id          ┆ category           ┆ flip   ┆ ref                     ┆ got              │
│ ---         ┆ ---                ┆ ---    ┆ ---                     ┆ ---              │
│ str  

## 4. Variant C — format hygiene

§3's verdict: the readmit breaks are intrinsic to the reframe's *form*, not to
"team" in the list — Bronny and Wemby still go null with it removed, and the
placeholder echo survives "answer with the player's name" too (it's the
`"Player Name"` literal in the tail the model copies). So the reframe family
fails the `readmit_affirm` floor (5/9 in every cut) and moves `true_toward`
down one, while being the only thing that moves the Luka trade cell
(33/38 `subject_not_target` with don't-guess). That is the ticket's
"partial win at best", and under the decision rule it is rejected; §7 records
the trade-off for the owner because the two floors it pays from are not
symmetric in production (a missed re-admission keeps a receipt dropped; a
missed null ships a misfiled one).

The floor-holding candidate is **don't-guess v2 alone** (§2c). Format hygiene
on top of it: the parser already recovers fenced JSON and takes the first
object, so this line's win condition isn't accuracy — it's output tokens at
scale and the end of the fence-plus-explanation pattern, whose "let me
reconsider" tail can make the first object the worse answer
(`receipt-m09` at baseline). Adopt if every floor holds and verdicts don't move.

In [10]:
# §4 — no-explanation tail on top of don't-guess v2; format metrics across runs.
TAIL_V2 = 'Respond with JSON only, no explanation: {{"t":"Player Name"|null,"c":0.0-1.0}}'

variant_c = run_variant(
    "c_noguess_noexplain", make_builder(lines=(PROD_TARGET_LINE, PROD_NULL_LINE, GUESS_LINE_V2), tail=TAIL_V2)
)
print(category_table(variant_c))
print(flips(variant_c, guess_v2).select("id", "category", "flip", "ref", "got"))
print("unresolved:", unresolved(variant_c).select("id", "got").rows())
print()
print(format_table({"baseline": baseline, "guess_v2": guess_v2, "C": variant_c}))

shape: (5, 5)
┌────────────────────┬───────┬────────┬───────┬──────────┐
│ category           ┆ acc   ┆ pinned ┆ delta ┆ floor_ok │
│ ---                ┆ ---   ┆ ---    ┆ ---   ┆ ---      │
│ str                ┆ str   ┆ str    ┆ i64   ┆ bool     │
╞════════════════════╪═══════╪════════╪═══════╪══════════╡
│ true_toward        ┆ 47/52 ┆ 49/52  ┆ -2    ┆ true     │
│ readmit_affirm     ┆ 6/9   ┆ 7/9    ┆ -1    ┆ true     │
│ wrong_player       ┆ 1/1   ┆ 1/1    ┆ 0     ┆ true     │
│ non_player         ┆ 9/11  ┆ 8/11   ┆ 1     ┆ true     │
│ subject_not_target ┆ 31/38 ┆ 24/38  ┆ 7     ┆ true     │
└────────────────────┴───────┴────────┴───────┴──────────┘
shape: (6, 5)
┌─────────────┬────────────────────┬────────┬───────────────────┬──────────────────────────────────┐
│ id          ┆ category           ┆ flip   ┆ ref               ┆ got                              │
│ ---         ┆ ---                ┆ ---    ┆ ---               ┆ ---                              │
│ str         ┆ str 

## 5. Reserve — the model ceiling

§4's verdict: **no format line.** The model fences its JSON on every row under
every prompt (`not_bare_json` 111/111 throughout), so "no explanation" doesn't
buy bare JSON; it halves truncations and cuts output a third, but moves
verdicts (4 broken, two of them name-form regressions), failing its own
adoption rule. Truncation is harmless anyway — zero invalid parses across
every run means the object always lands before the 75-token cap, so the cap
already acts as the cost bound. The structural fix is a harness change for
the pipeline PR, not a prompt line: an assistant prefill of `{` on Haiku 4.5
(prefill returns a 400 on the 4.6+ family, where structured outputs are the
replacement).

The reserve question was written for "the mention bucket doesn't move under
any form on Haiku". It did move (§3) — but only in a form that pays from
`readmit_affirm`. That reframes the question: is the *trade-off* the ceiling?
The verifier runs on ~10k rows (top-K pool + strata), so model cost is a few
dollars either way and the ticket's "the suite decides model + prompt" is
live. Two runs on Sonnet: `v0-draft` (the planned datapoint) and don't-guess
v2 (the floor-holding Haiku winner), scored against the same pinned picture.
Sonnet 5 takes no `temperature`, so these run at default sampling with
thinking disabled — single-run deltas here carry sampling noise the Haiku
runs don't.

In [11]:
# §5 — Sonnet on v0-draft and on don't-guess v2.
SONNET = "claude-sonnet-5"

sonnet_v0 = run_variant("sonnet_baseline", build_target_prompt, model=SONNET)
sonnet_guess = run_variant(
    "sonnet_noguess_v2",
    make_builder(lines=(PROD_TARGET_LINE, PROD_NULL_LINE, GUESS_LINE_V2)),
    model=SONNET,
)

for label, res, ref in [
    ("Sonnet v0-draft (vs Haiku control)", sonnet_v0, baseline),
    ("Sonnet + don't-guess v2 (vs Haiku don't-guess v2)", sonnet_guess, guess_v2),
]:
    print(f"===== {label} =====")
    print(category_table(res))
    print(flips(res, ref).select("id", "category", "flip", "ref", "got"))
    print("unresolved:", unresolved(res).select("id", "got").rows())
    print()
print(format_table({"haiku_guess_v2": guess_v2, "sonnet_v0": sonnet_v0, "sonnet_guess_v2": sonnet_guess}))

===== Sonnet v0-draft (vs Haiku control) =====
shape: (5, 5)
┌────────────────────┬───────┬────────┬───────┬──────────┐
│ category           ┆ acc   ┆ pinned ┆ delta ┆ floor_ok │
│ ---                ┆ ---   ┆ ---    ┆ ---   ┆ ---      │
│ str                ┆ str   ┆ str    ┆ i64   ┆ bool     │
╞════════════════════╪═══════╪════════╪═══════╪══════════╡
│ true_toward        ┆ 45/52 ┆ 49/52  ┆ -4    ┆ true     │
│ readmit_affirm     ┆ 7/9   ┆ 7/9    ┆ 0     ┆ true     │
│ wrong_player       ┆ 1/1   ┆ 1/1    ┆ 0     ┆ true     │
│ non_player         ┆ 9/11  ┆ 8/11   ┆ 1     ┆ true     │
│ subject_not_target ┆ 32/38 ┆ 24/38  ┆ 8     ┆ true     │
└────────────────────┴───────┴────────┴───────┴──────────┘
shape: (21, 5)
┌─────────────┬────────────────────┬────────┬─────────────────────────┬────────────────┐
│ id          ┆ category           ┆ flip   ┆ ref                     ┆ got            │
│ ---         ┆ ---                ┆ ---    ┆ ---                     ┆ ---            │
│ str   

### 5b. Sonnet across the variants; net correct as the line

Cost turns out not to separate the models: Sonnet's rate is 2× Haiku's, but
Haiku spends ~5× the output tokens on fences and explanations, so a 20k-row
verifier run is ~$6 on Haiku vs ~$7 on Sonnet (half that on the Batch API).
So Sonnet gets the same variants Haiku did — the reframe cuts and the null
list — and every run to date is put on one table.

The decision line, revisited with the owner: the `true_toward` veto was
borrowed from 06, where the regression category *was* the harm. Here the
five categories are the suite's split of one question, so the primary
measure is **net correct over the 111 cases**, with **no category floor
failing** as the guardrail (a floor failure is a systematic regression on
one failure mode, whatever the net says). Sonnet on `v0` is +5 net over the
control but still below Haiku + don't-guess v2 (+7) — the table below asks
whether any Sonnet variant clears that.

In [12]:
# §5b — Sonnet on the reframe / null-list variants; one table for every run.
sonnet_reframe = run_variant(
    "sonnet_b1_reframe", make_builder(ask=REFRAME_ASK, lines=(PROD_TARGET_LINE,)), model=SONNET
)
sonnet_reframe_guess = run_variant(
    "sonnet_b_reframe_noguess",
    make_builder(ask=REFRAME_ASK, lines=(PROD_TARGET_LINE, GUESS_LINE_V2)),
    model=SONNET,
)
sonnet_reframe2 = run_variant(
    "sonnet_b3_reframe2", make_builder(ask=REFRAME_ASK_V2, lines=(PROD_TARGET_LINE,)), model=SONNET
)
sonnet_reframe2_guess = run_variant(
    "sonnet_b3_reframe2_noguess",
    make_builder(ask=REFRAME_ASK_V2, lines=(PROD_TARGET_LINE, GUESS_LINE_V2)),
    model=SONNET,
)
sonnet_nulllist_guess = run_variant(
    "sonnet_b2_nulllist_noguess",
    make_builder(lines=(PROD_TARGET_LINE, NULL_LINE_V2, GUESS_LINE_V2)),
    model=SONNET,
)

ALL_RUNS = {
    "haiku control (v0)": baseline,
    "haiku A: don't-guess v2": guess_v2,
    "haiku A(v1 lines)": variant_a,
    "haiku name v2 alone": name_v2,
    "haiku B: reframe+guess": variant_b,
    "haiku B2: null-list+guess": variant_b2,
    "haiku B3: reframe2+guess": variant_b3,
    "haiku C: guess+no-explain": variant_c,
    "sonnet v0": sonnet_v0,
    "sonnet don't-guess v2": sonnet_guess,
    "sonnet reframe": sonnet_reframe,
    "sonnet reframe+guess": sonnet_reframe_guess,
    "sonnet reframe2": sonnet_reframe2,
    "sonnet reframe2+guess": sonnet_reframe2_guess,
    "sonnet null-list+guess": sonnet_nulllist_guess,
}


def summary_table(runs: dict[str, dict[str, dict]]) -> pl.DataFrame:
    """Every run on one table: per-category correct, net over 111, floors held."""
    rows = []
    for name, res in runs.items():
        tallies = target_accuracy_by_category(CASES, res, ALIAS_MAP)
        rows.append(
            {
                "run": name,
                **{cat[:7]: c for cat, (c, _) in sorted(tallies.items())},
                "net": sum(c for c, _ in tallies.values()),
                "floors_ok": all(c / t >= FLOORS[cat] for cat, (c, t) in tallies.items()),
            }
        )
    return pl.DataFrame(rows).sort("net", descending=True)


print(summary_table(ALL_RUNS))
for label, res in [("sonnet reframe2+guess", sonnet_reframe2_guess), ("sonnet reframe", sonnet_reframe)]:
    print(f"\n===== {label} (vs sonnet v0) =====")
    print(flips(res, sonnet_v0).select("id", "category", "flip", "ref", "got"))

shape: (15, 8)
┌───────────────────────────┬─────────┬─────────┬─────────┬─────────┬─────────┬─────┬───────────┐
│ run                       ┆ non_pla ┆ readmit ┆ subject ┆ true_to ┆ wrong_p ┆ net ┆ floors_ok │
│ ---                       ┆ ---     ┆ ---     ┆ ---     ┆ ---     ┆ ---     ┆ --- ┆ ---       │
│ str                       ┆ i64     ┆ i64     ┆ i64     ┆ i64     ┆ i64     ┆ i64 ┆ bool      │
╞═══════════════════════════╪═════════╪═════════╪═════════╪═════════╪═════════╪═════╪═══════════╡
│ sonnet reframe+guess      ┆ 9       ┆ 7       ┆ 34      ┆ 49      ┆ 1       ┆ 100 ┆ true      │
│ sonnet reframe            ┆ 9       ┆ 7       ┆ 34      ┆ 47      ┆ 1       ┆ 98  ┆ true      │
│ haiku B: reframe+guess    ┆ 10      ┆ 5       ┆ 34      ┆ 48      ┆ 0       ┆ 97  ┆ false     │
│ sonnet reframe2           ┆ 10      ┆ 7       ┆ 33      ┆ 46      ┆ 1       ┆ 97  ┆ true      │
│ sonnet reframe2+guess     ┆ 10      ┆ 7       ┆ 32      ┆ 47      ┆ 1       ┆ 97  ┆ true      │
│ son

## 6. Freeze candidate — Sonnet + reframe + don't-guess, 3-run confirmation

§5b's table answers the model question: **Sonnet with the §3 reframe and
don't-guess v2 is the only run that clears 100/111**, +11 net over the
control and +4 over the best Haiku option, with `true_toward` flat at 49,
every floor held, and the Luka trade rows null. The two things that sank the
reframe on Haiku — "team" taking the borderline readmit rows, the
placeholder echo — don't appear on Sonnet (`readmit_affirm` 7/9, same as the
control; bare JSON on every row). Sonnet 5 takes no `temperature`, so the
single-run picture carries sampling noise the Haiku runs didn't; three runs
settle which flips are stable prompt+model effects. The stable-flip table at
the bottom is the `target_cases.yaml` re-pin driver — every stable change of
correctness vs the pinned flags becomes a `known_miss` add/remove in the
freeze commit, and floors re-pin at worst-run-minus-one.

In [13]:
# §6 — freeze candidate: 3 runs of Sonnet + reframe + don't-guess v2 (run 1 is §5b's, cached).
FREEZE_BUILDER = make_builder(ask=REFRAME_ASK, lines=(PROD_TARGET_LINE, GUESS_LINE_V2))
FREEZE_MODEL = SONNET

runs = [run_variant("sonnet_b_reframe_noguess", FREEZE_BUILDER, run=k, model=FREEZE_MODEL) for k in (1, 2, 3)]
tallies = [target_accuracy_by_category(CASES, r, ALIAS_MAP) for r in runs]

print(
    pl.DataFrame(
        [
            {
                "category": cat,
                **{f"run{k}": f"{t[cat][0]}/{t[cat][1]}" for k, t in enumerate(tallies, 1)},
                "pinned": f"{PINNED[cat][0]}/{PINNED[cat][1]}",
                "floor_ok_all": all(t[cat][0] / t[cat][1] >= FLOORS[cat] for t in tallies),
            }
            for cat in sorted(FLOORS)
        ]
    )
)
print("net:", [sum(c for c, _ in t.values()) for t in tallies])
print(format_table({f"run{k}": r for k, r in enumerate(runs, 1)}))

_unstable = [
    {
        "id": c.id, "category": c.category, "expected": str(c.expected_target),
        "got": " / ".join(str(r[c.id]["t"]) for r in runs), "text": c.text[:50].replace("\n", " "),
    }
    for c in CASES
    if len({correct(r, c.id) for r in runs}) > 1
]
print(f"\nunstable cases across 3 runs: {len(_unstable)}")
print(pl.DataFrame(_unstable, schema={"id": str, "category": str, "expected": str, "got": str, "text": str}))

# Stable correctness changes vs the pinned flags -> the target_cases.yaml re-pin.
_repin = [
    {
        "id": c.id, "category": c.category, "expected": str(c.expected_target),
        "got": str(runs[0][c.id]["t"]),
        "action": "remove known_miss" if c.known_miss else "add known_miss",
        "text": c.text[:50].replace("\n", " "),
    }
    for c in CASES
    if len({correct(r, c.id) for r in runs}) == 1 and correct(runs[0], c.id) == c.known_miss
]
print(f"\nstable flips vs pinned flags (target_cases.yaml re-pin): {len(_repin)}")
print(pl.DataFrame(_repin, schema={"id": str, "category": str, "expected": str, "got": str, "action": str, "text": str}))

shape: (5, 6)
┌────────────────────┬───────┬───────┬───────┬────────┬──────────────┐
│ category           ┆ run1  ┆ run2  ┆ run3  ┆ pinned ┆ floor_ok_all │
│ ---                ┆ ---   ┆ ---   ┆ ---   ┆ ---    ┆ ---          │
│ str                ┆ str   ┆ str   ┆ str   ┆ str    ┆ bool         │
╞════════════════════╪═══════╪═══════╪═══════╪════════╪══════════════╡
│ non_player         ┆ 9/11  ┆ 9/11  ┆ 9/11  ┆ 8/11   ┆ true         │
│ readmit_affirm     ┆ 7/9   ┆ 7/9   ┆ 6/9   ┆ 7/9    ┆ true         │
│ subject_not_target ┆ 34/38 ┆ 33/38 ┆ 32/38 ┆ 24/38  ┆ true         │
│ true_toward        ┆ 49/52 ┆ 50/52 ┆ 50/52 ┆ 49/52  ┆ true         │
│ wrong_player       ┆ 1/1   ┆ 1/1   ┆ 1/1   ┆ 1/1    ┆ true         │
└────────────────────┴───────┴───────┴───────┴────────┴──────────────┘
net: [100, 100, 98]
shape: (3, 5)
┌──────┬─────────┬───────────┬───────────────┬────────────┐
│ run  ┆ invalid ┆ truncated ┆ not_bare_json ┆ mean_chars │
│ ---  ┆ ---     ┆ ---       ┆ ---           ┆ --- 

### 6b. Price of the full run

Official numbers rather than chars-over-four: input tokens from the API's
`count_tokens` endpoint (the real tokenizer, free) on the frozen prompt for
all 111 cases, output tokens the same way on the cached §6 responses, and the
Haiku don't-guess runner-up alongside for the comparison the model decision
rests on. The suite's bodies are mined receipts under the same 500-char cap
the pool applies, so per-row tokens transfer.

The pool is the one open input: top-K per player × polar sentiment plus the
two strata, with K sized by dry run in the pipeline PR. The shipped
`comment_samples` has 442 polar cells (345 full at N=10), so `442 × K` bounds
the top-K pool from above; the strata are assumed at 1,000 rows each. The
production run goes through the Batch API, so the batch column is the
number that lands in `state.json`; list rates are the sync (notebook) price.

In [14]:
# §6b — per-row tokens via count_tokens; run price as a function of K.
import anthropic

_client = anthropic.Anthropic()
HAIKU = TARGET_MODEL
RATES = {  # $ per MTok (input, output); batch = half list
    HAIKU: {"list": (1.00, 5.00), "batch": (0.50, 2.50)},
    SONNET: {"list": (2.00, 10.00), "batch": (1.00, 5.00)},
}
HAIKU_RUNNER_UP = make_builder(lines=(PROD_TARGET_LINE, PROD_NULL_LINE, GUESS_LINE_V2))


def count(model: str, text: str) -> int:
    """Tokens for `text` as a single user message, per the API tokenizer."""
    return _client.messages.count_tokens(
        model=model, messages=[{"role": "user", "content": text}]
    ).input_tokens


def per_row_tokens(model: str, builder: Builder, results: dict[str, dict]) -> tuple[float, float]:
    """Mean (input, output) tokens per case; output counted as bare text minus message framing."""
    framing = count(model, "x") - 1
    inp = sum(count(model, builder(c.text, c.sentiment)) for c in CASES) / len(CASES)
    out = sum(count(model, r["raw"]) - framing for r in results.values()) / len(results)
    return inp, out


tok = {
    "sonnet freeze": (SONNET, *per_row_tokens(SONNET, FREEZE_BUILDER, runs[0])),
    "haiku runner-up": (HAIKU, *per_row_tokens(HAIKU, HAIKU_RUNNER_UP, guess_v2)),
}
print(pl.DataFrame([{"candidate": k, "model": m, "in_tok/row": round(i, 1), "out_tok/row": round(o, 1)} for k, (m, i, o) in tok.items()]))

POLAR_CELLS = 442
STRATA_ROWS = 2 * 1_000
rows_for = lambda k: POLAR_CELLS * k + STRATA_ROWS

print(
    pl.DataFrame(
        [
            {
                "K": k,
                "rows (upper bound)": rows_for(k),
                **{
                    f"{name} {tier}": round(rows_for(k) * (i * ri + o * ro) / 1e6, 2)
                    for name, (m, i, o) in tok.items()
                    for tier, (ri, ro) in RATES[m].items()
                },
            }
            for k in (10, 20, 30, 50)
        ]
    )
)

shape: (2, 4)
┌─────────────────┬───────────────────────────┬────────────┬─────────────┐
│ candidate       ┆ model                     ┆ in_tok/row ┆ out_tok/row │
│ ---             ┆ ---                       ┆ ---        ┆ ---         │
│ str             ┆ str                       ┆ f64        ┆ f64         │
╞═════════════════╪═══════════════════════════╪════════════╪═════════════╡
│ sonnet freeze   ┆ claude-sonnet-5           ┆ 228.4      ┆ 16.5        │
│ haiku runner-up ┆ claude-haiku-4-5-20251001 ┆ 170.4      ┆ 38.5        │
└─────────────────┴───────────────────────────┴────────────┴─────────────┘
shape: (4, 6)
┌─────┬────────────────────┬────────────────────┬─────────────────────┬──────────────────────┬───────────────────────┐
│ K   ┆ rows (upper bound) ┆ sonnet freeze list ┆ sonnet freeze batch ┆ haiku runner-up list ┆ haiku runner-up batch │
│ --- ┆ ---                ┆ ---                ┆ ---                 ┆ ---                  ┆ ---                   │
│ i64 ┆ i64    

## 7. Verdict & freeze

| Candidate | Decision | Evidence (artifacts in `EXP_DIR`, `target_*`) |
|---|---|---|
| Name as written | **reject** | Both wordings induce the mention-as-target failure (`nullp-m01`, `receipt-m13`, `random-m23` regress) and copy epithets (`Luka Flopbitch`); `subject_not_target` 23/38 alone |
| Don't guess, v1 wording | **reject** | "named or unambiguously described … do not guess" reads as *when in doubt, null*: nulls plainly named players, `true_toward` −3 |
| Don't guess, v2 ("Never name a player who does not appear in the comment.") | **adopt** | +7 net on Haiku with every floor held; +2 net on top of the reframe on Sonnet; captures the name-as-written win by itself (`Brooks`, `CMB`) |
| Reframed ask (non-player answers first-class) | **adopt, on Sonnet** | The only form that moves the Luka trade cell; on Haiku it pays from `readmit_affirm` (5/9 in every cut) and echoes the placeholder — neither appears on Sonnet |
| Extended null list | **reject** | Holds floors but doesn't reach the trade cell; the win is the question's form, not list coverage |
| "No explanation" tail | **reject** | Fenced JSON on 111/111 rows regardless; halves truncation but moves verdicts; truncation is harmless (0 invalid parses in every run) — the 75-token cap is the cost bound |
| Model: Sonnet 5 | **adopt** | Sonnet + reframe + don't-guess is the only run at 100/111: net 100/100/98 ×3, floors held ×3, `true_toward` 49/50/50 vs 49 pinned, bare JSON on 332/333 rows. Haiku's best floor-holding option is 96 |

**Decision rule, amended with the owner (2026-09-07):** the primary measure is
**net correct over the suite**; the guardrail is **no category floor fails**.
The `true_toward` veto was borrowed from 06, where the regression category was
the harm; here the five categories are one question split by failure mode, and
the ticket's own posture (thin visibly, never misfile) weights a dropped
receipt below a misfiled one. Cost does not separate the models (§6b): the
production run is ~$2–5 on the Batch API at any plausible K.

**Frozen verifier:** Sonnet 5, no `temperature` (rejected by the model),
thinking disabled, `max_tokens` 75; the prompt printed below;
`TARGET_PROMPT_VERSION` `v1`.

**Known limitations, accepted for v1:** two-target rows null under the
single-target contract (`receipt-m61`, `random-m05` now passes by naming the
first); a bare list of players nulls (`random-m21`); "Fox sold it" nulls at
low confidence (`receipt-m28`); the model sometimes emits diacritics the
config doesn't carry (`Luka Dončić`, `Donté DiVincenzo`) — a resolver
normalization question, timed with the `resolve_player` decision, not a
prompt one. Four cases wobble across runs at default sampling and are flagged.

**Freeze sequence** (this notebook executes first, then):

1. `pipeline/targets.py` — `TARGET_MODEL`, request params, template, version;
   sha pin re-pinned in `tests/unit/test_targets.py`.
2. `tests/eval/target_cases.yaml` — a case is expected-correct iff it passed
   all three §6 runs: 15 `known_miss` flags removed, 4 added, 3 unstable
   flagged; floors at worst-run-minus-one (`non_player` 0.72,
   `readmit_affirm` 0.55, `subject_not_target` 0.81, `true_toward` 0.92,
   `wrong_player` exact).
3. `uv run pytest -m eval tests/eval/test_target_eval.py --maxfail=0 -rxX`
   — green under the new verifier, flags, and floors.
4. After the freeze this notebook is a closed record: the §2 faithfulness
   assert and the `TARGET_TEMPERATURE` import fail against the new module *by
   design*. The executed outputs and `EXP_DIR` artifacts are the evidence; do
   not re-execute.

**For the pipeline PR:** the batch orchestrator must build verifier requests
without `temperature` and with thinking disabled; the sync dry run that sizes
K uses the harness runner; both transports write the downloader's five-field
rows so the sidecar builder is transport-agnostic.

In [15]:
# §7 — the frozen prompt, printed from the winning builder.
frozen_prompt = FREEZE_BUILDER("{comment_body}", "neg").replace("negative", "{sentiment_word}")
print(frozen_prompt)
print(f"\nmodel: {FREEZE_MODEL}   freeze sha: {prompt_sha(FREEZE_BUILDER)}  "
      f"(matches target_sonnet_b_reframe_noguess_run[1-3].json artifacts)")

This r/NBA comment was labeled {sentiment_word}. Is that {sentiment_word} sentiment aimed at an NBA player, or at a team, front office, coach, referees, a decision, or a play? If at a player, name them; otherwise answer null.
The target is the player being praised or criticized - not a player who is merely mentioned, sympathized with, or the subject of someone else's decision.
Never name a player who does not appear in the comment.

Comment: {comment_body}

Respond ONLY with JSON: {"t":"Player Name"|null,"c":0.0-1.0}

model: claude-sonnet-5   freeze sha: e3c242175bc5fe95  (matches target_sonnet_b_reframe_noguess_run[1-3].json artifacts)
